## Token Usage & Cost Analysis

Token counts observed from a single run for **The Fool** (tarot card) using `qwen3.5:0.8b` locally.
Costs are estimated for cloud APIs if the same pipeline were run there instead.

In [ ]:
import pandas as pd

# ── Token counts from one observed run (The Fool, qwen3.5:0.8b) ──────────────
# search_sources uses web search only — no LLM tokens
steps = {
    "analyze_sources": {"calls": 1, "input": 4_096, "output": 597},
    "generate_document": {"calls": 1, "input": 903, "output": 1_376},
    "analyze_document": {"calls": 1, "input": 1_518, "output": 207},
    "generate_affirmations": {"calls": 16, "input": 10_106, "output": 871},
    # 48 validation calls total (16 grammars × 3 affirmations).
    # Output was truncated in the log after grammar 8/16 (24 calls observed).
    # Extrapolated: avg 273 input & 129 output per call × 48 calls.
    "validate_affirmations": {"calls": 48, "input": 13_090, "output": 6_206},
}

df_steps = pd.DataFrame(steps).T.astype(int)
df_steps["total"] = df_steps["input"] + df_steps["output"]
df_steps.index.name = "step"

total_input = df_steps["input"].sum()
total_output = df_steps["output"].sum()
total_tokens = df_steps["total"].sum()

print(f"{'Step':<30} {'Calls':>6} {'Input':>8} {'Output':>8} {'Total':>8}")
print("-" * 66)
for step, row in df_steps.iterrows():
    print(
        f"{step:<30} {int(row['calls']):>6} {int(row['input']):>8,} {int(row['output']):>8,} {int(row['total']):>8,}"
    )
print("-" * 66)
print(
    f"{'TOTAL':<30} {int(df_steps['calls'].sum()):>6} {total_input:>8,} {total_output:>8,} {total_tokens:>8,}"
)

In [ ]:
from src.subjects import (
    TAROT_CARDS,
)

# All subject lists — add more as they are created
all_subjects = TAROT_CARDS
total_subjects = len(all_subjects)
print(f"Total subjects: {total_subjects}")

# ── Cloud API pricing  ($ per 1 M tokens, as of April 2026) ───────────────────
# Sources: https://openrouter.ai  (confirmed April 2026)
models = {
    # label                            $/1M input   $/1M output
    # ── OpenAI GPT-5.4 ──────────────────────────────────────
    "GPT-5.4": (2.50, 15.00),
    "GPT-5.4 mini": (0.75, 4.50),
    "GPT-5.4 nano": (0.20, 1.25),
    # ── Google Gemini 2.5 ───────────────────────────────────
    "Gemini 2.5 Pro": (1.25, 10.00),
    "Gemini 2.5 Flash": (0.30, 2.50),
    "Gemini 2.5 Flash-Lite": (0.10, 0.40),
    # ── Google Gemini 3 / 3.1 ───────────────────────────────
    "Gemini 3 Flash Preview": (0.50, 3.00),
    "Gemini 3.1 Flash Lite Preview": (0.25, 1.50),
    "Gemini 3.1 Pro Preview": (2.00, 12.00),
    # ── Anthropic Claude 4.x ────────────────────────────────
    "Claude Haiku 4.5": (1.00, 5.00),
    "Claude Sonnet 4.5 / 4.6": (3.00, 15.00),
    "Claude Opus 4.6": (5.00, 25.00),
    "Claude Opus 4": (15.00, 75.00),
}

rows = []
for name, (price_in, price_out) in models.items():
    cost_per_subject = total_input / 1_000_000 * price_in + total_output / 1_000_000 * price_out
    cost_total = cost_per_subject * total_subjects
    rows.append(
        {
            "Model": name,
            "$/1M input": f"${price_in:.2f}",
            "$/1M output": f"${price_out:.2f}",
            "Cost/subject": f"${cost_per_subject:.4f}",
            f"Cost × {total_subjects} subjects": f"${cost_total:.3f}",
        }
    )

df_costs = pd.DataFrame(rows)
print("\n" + df_costs.to_string(index=False))
print(
    f"\nTokens per subject: {total_input:,} input + {total_output:,} output = {total_tokens:,} total"
)
print("Note: validate_affirmations output is extrapolated from 24/48 observed calls.")